# Votos e votações - Senado/Brasil

Usa o XML anual oficial. A UF vem do próprio registro de voto.

O mapa histórico só entra como fallback caso algum retorno não traga a UF do parlamentar.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import shutil
import tempfile
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "senado"
LEGISLATURA = 57
ANOS = [2023, 2024, 2025, 2026]

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")
TMP = Path(tempfile.mkdtemp(prefix="pi_ii_bronze_senado_"))

FORCAR_RECOLETA = False

LEGIS_BASE = "https://legis.senado.leg.br/dadosabertos"
ADM_BASE = "https://adm.senado.gov.br/adm-dadosabertos"

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "PI-II-Univesp-Bronze-Senado-Brasil/2.1",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def pasta_uf(uf):
    return ROOT / uf.lower()

def normalizar_nome(valor):
    return re.sub(r"[^a-z0-9]", "", str(valor).lower())

def api_get(url, params=None, timeout=(30, 180), accept="application/json"):
    headers = {"Accept": accept}
    r = session.get(url, params=params, timeout=timeout, headers=headers)
    r.raise_for_status()
    if "json" in (r.headers.get("content-type") or "").lower() or accept == "application/json":
        return r.json()
    return r.text

def baixar(url, destino, timeout=(30, 900)):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    parcial = destino.with_suffix(destino.suffix + ".part")

    if parcial.exists():
        parcial.unlink()

    print("Baixando:", url)

    with session.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()

        total = int(r.headers.get("content-length") or 0)
        recebido = 0
        ultimo_print = time.time()

        with parcial.open("wb") as f:
            for bloco in r.iter_content(chunk_size=1024 * 1024):
                if not bloco:
                    continue

                f.write(bloco)
                recebido += len(bloco)

                if time.time() - ultimo_print >= 5:
                    if total:
                        print(
                            f"  {recebido / 1024**2:.1f} MB / "
                            f"{total / 1024**2:.1f} MB"
                        )
                    else:
                        print(f"  {recebido / 1024**2:.1f} MB")
                    ultimo_print = time.time()

    parcial.replace(destino)
    print(f"Concluído: {destino.name}")
    return destino

def extrair_por_chave(obj, chave):
    alvo = normalizar_nome(chave)
    encontrados = []

    def walk(x):
        if isinstance(x, dict):
            for k, v in x.items():
                if normalizar_nome(k) == alvo:
                    if isinstance(v, list):
                        encontrados.extend([i for i in v if isinstance(i, dict)])
                    elif isinstance(v, dict):
                        encontrados.append(v)
                walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)

    walk(obj)

    unicos = []
    vistos = set()

    for item in encontrados:
        canon = json.dumps(item, ensure_ascii=False, sort_keys=True, default=str)
        if canon not in vistos:
            vistos.add(canon)
            unicos.append(item)

    return unicos

def maior_lista_de_dicts(obj):
    listas = []

    def walk(x):
        if isinstance(x, list):
            if x and all(isinstance(i, dict) for i in x):
                listas.append(x)
            for item in x:
                walk(item)
        elif isinstance(x, dict):
            for v in x.values():
                walk(v)

    walk(obj)

    if not listas:
        return []

    return max(listas, key=len)

def achar_coluna(df, candidatos=None, contem_todos=None):
    candidatos = candidatos or []
    mapa = {normalizar_nome(c): c for c in df.columns}

    for nome in candidatos:
        chave = normalizar_nome(nome)
        if chave in mapa:
            return mapa[chave]

    if contem_todos:
        termos = [normalizar_nome(x) for x in contem_todos]
        for c in df.columns:
            nc = normalizar_nome(c)
            if all(t in nc for t in termos):
                return c

    return None

def detectar_ano(df):
    candidatos = [
        c for c in df.columns
        if normalizar_nome(c) in {
            "anomateria", "materiaano", "anoproposicao", "ano"
        }
        or normalizar_nome(c).endswith("anomateria")
    ]

    for c in candidatos:
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            return s.astype("Int64")

    datas = [
        c for c in df.columns
        if "data" in normalizar_nome(c)
    ]

    for c in datas:
        dt = pd.to_datetime(df[c], errors="coerce", dayfirst=True)
        if dt.notna().any():
            return dt.dt.year.astype("Int64")

    return pd.Series([pd.NA] * len(df), index=df.index, dtype="Int64")


def carregar_mapa_historico():
    path = ROOT / "_senadores_exercicios_brasil.csv"

    if not path.exists():
        raise FileNotFoundError(
            "Mapa histórico não encontrado. Execute primeiro o notebook 00."
        )

    df = pd.read_csv(
        path,
        sep=";",
        dtype=str,
        keep_default_na=False,
    )

    obrigatorias = {"codigo_parlamentar", "uf"}

    if not obrigatorias.issubset(df.columns):
        raise RuntimeError(
            "O mapa histórico não tem as colunas esperadas."
        )

    mapa = (
        df[["codigo_parlamentar", "uf"]]
        .assign(
            codigo_parlamentar=lambda x: (
                x["codigo_parlamentar"].astype(str).str.strip()
            ),
            uf=lambda x: (
                x["uf"].astype(str).str.strip().str.upper()
            ),
        )
        .drop_duplicates(subset=["codigo_parlamentar"])
        .set_index("codigo_parlamentar")["uf"]
        .to_dict()
    )

    return mapa


def salvar_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep=";", index=False, encoding="utf-8")
    print(f"Salvo: {path} | {len(df):,}")

def salvar_manifesto(nome, payload):
    payload = dict(payload)
    payload["gerado_em_utc"] = utc_now()

    path = ROOT / f"_manifest_{nome}_brasil.json"
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Manifesto:", path)

print("Bronze Senado:", ROOT)
print("Legislatura:", LEGISLATURA)
print("Anos:", ANOS)


In [ ]:
import calendar
import xml.etree.ElementTree as ET

# usado só como fallback quando o voto não traz UF
cod_para_uf = carregar_mapa_historico()

print("Códigos no mapa histórico:", len(cod_para_uf))


In [ ]:
def tag_local(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def flatten_xml(node, prefix=""):
    out = {}

    for child in list(node):
        nome = tag_local(child.tag)
        chave = f"{prefix}_{nome}" if prefix else nome

        if list(child):
            nested = flatten_xml(child, chave)
            out.update(nested)
        else:
            out[chave] = (child.text or "").strip()

    return out

def parse_votacoes_xml(path, ano):
    tree = ET.parse(path)
    root = tree.getroot()

    votacoes = []
    votos = []

    # usa só elementos Votacao que tenham CodigoSessaoVotacao como filho direto
    for node in root.iter():
        if tag_local(node.tag) != "Votacao":
            continue

        filhos = {tag_local(c.tag): c for c in list(node)}

        if "CodigoSessaoVotacao" not in filhos:
            continue

        meta = {}

        for child in list(node):
            nome = tag_local(child.tag)

            if nome.lower() in {"votos", "votoparlamentar", "votosparlamentares"}:
                continue

            if list(child):
                continue

            meta[nome] = (child.text or "").strip()

        meta["_ano_recorte"] = ano
        votacoes.append(meta)

        # pega VotoParlamentar dentro desta votação
        for voto_node in node.iter():
            if tag_local(voto_node.tag) != "VotoParlamentar":
                continue

            row = flatten_xml(voto_node)
            row["_ano_recorte"] = ano
            row["_CodigoSessaoVotacao"] = meta.get("CodigoSessaoVotacao", "")
            row["_DataSessao"] = meta.get("DataSessao", "")
            row["_CodigoMateria"] = meta.get("CodigoMateria", "")
            row["_SiglaMateria"] = meta.get("SiglaMateria", "")
            row["_NumeroMateria"] = meta.get("NumeroMateria", "")
            row["_AnoMateria"] = meta.get("AnoMateria", "")
            votos.append(row)

    return pd.DataFrame(votacoes), pd.DataFrame(votos)

def parse_votacoes_json(payload, ano):
    regs = extrair_por_chave(payload, "Votacao")
    votacoes = []
    votos = []

    for reg in regs:
        flat_vot = pd.json_normalize([reg], sep="_")

        # tenta localizar lista(s) de voto antes de transformar
        votos_reg = extrair_por_chave(reg, "VotoParlamentar")

        # mantém a votação sem estruturas de lista em JSON serializado
        linha = {}
        for c in flat_vot.columns:
            valor = flat_vot.iloc[0][c]
            if not isinstance(valor, (list, dict)):
                linha[c] = valor

        linha["_ano_recorte"] = ano
        votacoes.append(linha)

        codigo_vot = ""
        for k, v in linha.items():
            if "codigosessaovotacao" in normalizar_nome(k):
                codigo_vot = str(v)
                break

        for voto in votos_reg:
            row = pd.json_normalize([voto], sep="_")
            row["_ano_recorte"] = ano
            row["_CodigoSessaoVotacao"] = codigo_vot
            votos.append(row.iloc[0].to_dict())

    return pd.DataFrame(votacoes), pd.DataFrame(votos)

def obter_votacoes_ano(ano):
    xml_url = (
        f"{LEGIS_BASE}/dados/ListaVotacoes{ano}.xml"
    )
    xml_path = TMP / f"ListaVotacoes{ano}.xml"

    try:
        baixar(xml_url, xml_path)
        votacoes, votos = parse_votacoes_xml(xml_path, ano)

        if not votos.empty:
            print("Fonte usada: XML anual")
            return votacoes, votos, "xml_anual"

    except Exception as exc:
        print("XML anual não funcionou:", exc)

    # fallback: 6 janelas bimestrais
    todas_votacoes = []
    todos_votos = []

    for mes_inicio in [1, 3, 5, 7, 9, 11]:
        mes_fim = min(mes_inicio + 1, 12)

        inicio = f"{ano}{mes_inicio:02d}01"
        ultimo_dia = calendar.monthrange(ano, mes_fim)[1]
        fim = f"{ano}{mes_fim:02d}{ultimo_dia:02d}"

        url = (
            f"{LEGIS_BASE}/plenario/lista/votacao/"
            f"{inicio}/{fim}.json"
        )

        try:
            payload = api_get(url)
        except requests.HTTPError as exc:
            if exc.response is not None and exc.response.status_code == 404:
                continue
            raise

        vot, votos = parse_votacoes_json(payload, ano)

        if not vot.empty:
            todas_votacoes.append(vot)
        if not votos.empty:
            todos_votos.append(votos)

    votacoes = (
        pd.concat(todas_votacoes, ignore_index=True, sort=False)
        if todas_votacoes else pd.DataFrame()
    )
    votos = (
        pd.concat(todos_votos, ignore_index=True, sort=False)
        if todos_votos else pd.DataFrame()
    )

    print("Fonte usada: web service bimestral")
    return votacoes, votos, "webservice_bimestral"


In [ ]:
resumo = []

for ano in ANOS:
    print("\n" + "=" * 70)
    print("ANO", ano)

    votacoes, votos, fonte = obter_votacoes_ano(ano)

    if votos.empty:
        raise RuntimeError(
            f"Não encontrei votos nominais para {ano}."
        )

    uf_voto_col = achar_coluna(
        votos,
        [
            "SiglaUf",
            "UfParlamentar",
            "IdentificacaoParlamentar_UfParlamentar",
        ],
        contem_todos=["uf"],
    )

    cod_voto_col = achar_coluna(
        votos,
        [
            "CodigoParlamentar",
            "IdentificacaoParlamentar_CodigoParlamentar",
        ],
        contem_todos=["codigo", "parlamentar"],
    )

    if uf_voto_col:
        votos["_uf_recorte"] = (
            votos[uf_voto_col]
            .astype(str)
            .str.strip()
            .str.upper()
        )
    elif cod_voto_col:
        votos["_uf_recorte"] = (
            votos[cod_voto_col]
            .astype(str)
            .str.strip()
            .map(cod_para_uf)
            .fillna("")
        )
    else:
        raise RuntimeError(
            f"Não encontrei UF nem código parlamentar nos votos de {ano}. "
            f"Colunas: {list(votos.columns)}"
        )

    id_vot_col = achar_coluna(
        votos,
        ["_CodigoSessaoVotacao", "CodigoSessaoVotacao"],
        contem_todos=["codigo", "sessao", "votacao"],
    )

    id_votacoes_col = achar_coluna(
        votacoes,
        ["CodigoSessaoVotacao", "_CodigoSessaoVotacao"],
        contem_todos=["codigo", "sessao", "votacao"],
    )

    if not id_vot_col or not id_votacoes_col:
        raise RuntimeError(
            f"Não encontrei o ID da votação em {ano}."
        )

    cont_votos = {}
    cont_votacoes = {}

    for uf in UFS:
        votos_uf = votos[votos["_uf_recorte"].eq(uf)].copy()

        salvar_csv(
            votos_uf,
            pasta_uf(uf) / f"votos_{ano}.csv",
        )

        ids = set(
            votos_uf[id_vot_col].astype(str).str.strip()
        )

        vot_uf = votacoes[
            votacoes[id_votacoes_col].astype(str).str.strip().isin(ids)
        ].copy()

        salvar_csv(
            vot_uf,
            pasta_uf(uf) / f"votacoes_{ano}.csv",
        )

        cont_votos[uf] = len(votos_uf)
        cont_votacoes[uf] = len(vot_uf)

    votos_sem_uf = votos[
        ~votos["_uf_recorte"].isin(UFS)
    ].copy()

    salvar_csv(
        votos_sem_uf,
        ROOT / f"_votos_nao_distribuidos_{ano}.csv",
    )

    resumo.append({
        "ano": ano,
        "status": "ok",
        "fonte": fonte,
        "votos_total": len(votos),
        "votacoes_total": len(votacoes),
        "votos_nao_distribuidos": len(votos_sem_uf),
        "votos_por_uf": cont_votos,
        "votacoes_por_uf": cont_votacoes,
    })

    print(
        "Votos:", len(votos),
        "| votações:", len(votacoes),
        "| votos sem UF:", len(votos_sem_uf),
    )

salvar_manifesto("votacoes", {"resumo": resumo})

display(pd.DataFrame([
    {
        "ano": x["ano"],
        "fonte": x["fonte"],
        "votos_total": x["votos_total"],
        "votacoes_total": x["votacoes_total"],
        "votos_nao_distribuidos": x["votos_nao_distribuidos"],
    }
    for x in resumo
]))
